# GP Analysis

This notebook reads in the input datasets used to run the routing tool for projects relevant to GPs, and produces outputs that can be used for further analysis.

* Load LSOA PWC and population data, GP practice and branch data, and shapefiles for LSOAs, ICBs and parliamentary constituencies.
* For driving and walking respectively, collect one row per LSOA, showing how many GPs are within 5, 10, 15, etc. minutes of each LSOA's PWC, as well as details about the closest GP (time-wise).
* For driving and walking respectively, produce population-weighted averages of ICB and parliamentary constituency summary statistics - using lookups of LSOAs to ICBs and LSOAs to constituencies.
* Plot basic maps of some summary statistics, to act as demonstrations.
* Save datasets to the `output` folder for exporting to Sharepoint.

In [0]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib import colors

from src.datavis import plot_map
from src.read_write_data import load_config, read_in_route, write_dataset
from src.stats import (
    aggregate_stats_by_geography,
    aggregate_summary_upto_larger_geography,
    bin_values,
    summarise_stats
)

config: dict = load_config("../../config.yml")

## Read in data

In [0]:
lsoa_pwcs: pd.DataFrame = (
    pd.read_csv(config["lsoa_pwc_clean_data_file_path"])
    .merge(
        pd.read_csv(config["lsoa_rural_urban_classification"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA21CD",
        validate="one_to_one",
    )
    .merge(
        pd.read_csv(config["lsoa_population_data"]),
        how="left",
        left_on="unique_code",
        right_on="LSOA 2021 Code"
    )
    .rename(columns={"Total": "population"})
    .assign(population=lambda df: df.population.str.replace(',', '').astype(int))
)[
    ["unique_code", "lat", "lon", "LSOA21NM", "RUC21CD", "RUC21NM", "Urban_rural_flag", "population"]
]
lsoa_pwcs.head(1)

In [0]:
gps: pd.DataFrame = (
    pd.read_csv(config["gps_clean"])
    .rename(columns={"name": "gp_name"})
    .drop(columns=["is_dispensing_gp", "number_of_dispensing_gps"])
)
gps.head(1)

In [0]:
lsoa_to_icb_df: pd.DataFrame = pd.read_csv(config["lsoa_to_icb"])
lsoa_to_icb_df.head(1)

In [0]:
icb_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/ICB_APR_2023_EN_BGC.shp"
)
icb_boundaries_en.head(1)

In [0]:
lsoa_to_constituency_df: pd.DataFrame = gpd.read_file(
    "../../input/map/LSOA_(2021)_to_Westminster_Parliamentary_Constituencies_(July_2024)_Best_Fit_Lookup_in_EW.shp"
)[["LSOA21CD", "PCON25CD", "PCON25NM"]].rename(columns={"PCON25CD": "PCON24CD", "PCON25NM": "PCON24NM"})
lsoa_to_constituency_df.head(1)

In [0]:
constituency_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/PCON_JULY_2024_UK_BFE.shp"
).query("PCON24CD.str[0] == 'E'")
constituency_boundaries_en.head(1)

In [0]:
lsoa_boundaries_en: gpd.GeoDataFrame = gpd.read_file(
    "../../input/map/LSOA_2021_EW_BSC_V4.shp"
).query("LSOA21CD.str[0] == 'E'")
lsoa_boundaries_en.head(1)

## Cars

In [0]:
lsoa_gp_car = read_in_route(
    config["database_path"], "lsoa_pwcs", "gps", "car"
)
lsoa_gp_car_stats: pd.DataFrame = summarise_stats(
    lsoa_gp_car, lsoa_pwcs, gps, [5, 10, 15, 20, 30]
)
lsoa_gp_car_stats

In [0]:
plot_map(
    lsoa_gp_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_15_minutes",
    "LSOA21CD",
    higher_is_worse=False,
    #norm=colors.TwoSlopeNorm(vcenter=30, vmin=0, vmax=50),
)
plot_map(
    lsoa_gp_car_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    #norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
plot_map(
    lsoa_gp_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_15_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_gp_car_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_gp_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_15_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_gp_car_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
(
    bin_values(lsoa_gp_car_stats, "min_time", [0, 5, 10, 15, 20, 25, 30])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
(
    bin_values(lsoa_gp_car, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

## Walking

In [0]:
lsoa_gp_foot = read_in_route(
    config["database_path"], "lsoa_pwcs", "gps", "foot"
)
lsoa_gp_foot_stats: pd.DataFrame = summarise_stats(
    lsoa_gp_foot, lsoa_pwcs, gps, [5, 10, 15, 20, 30]
)
lsoa_gp_foot_stats

In [0]:
aggregate_summary_upto_larger_geography(
    lsoa_gp_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
)

In [0]:
plot_map(
    lsoa_gp_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "number_of_locations_within_30_minutes",
    "LSOA21CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_gp_foot_stats,
    lsoa_pwcs,
    lsoa_boundaries_en,
    "min_time",
    "LSOA21CD",
    higher_is_worse=True,
    norm=colors.TwoSlopeNorm(vcenter=10, vmin=0, vmax=30),
)

In [0]:
plot_map(
    lsoa_gp_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "number_of_locations_within_30_minutes",
    "ICB23CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_gp_foot_stats,
    lsoa_to_icb_df,
    icb_boundaries_en,
    "min_time",
    "ICB23CD",
    higher_is_worse=True,
)

In [0]:
plot_map(
    lsoa_gp_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "number_of_locations_within_30_minutes",
    "PCON24CD",
    higher_is_worse=False,
)
plot_map(
    lsoa_gp_foot_stats,
    lsoa_to_constituency_df,
    constituency_boundaries_en,
    "min_time",
    "PCON24CD",
    higher_is_worse=True,
    # norm=colors.TwoSlopeNorm(vcenter=5, vmin=1, vmax=9)
)

In [0]:
(
    bin_values(
        lsoa_gp_foot_stats,
        "min_time",
        [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420])
    .groupby(["min_time_binned"])
    .agg({"population": sum})
    .reset_index()
    .query("population > 0")
)

In [0]:
(
    bin_values(lsoa_gp_foot, "time", [0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100, 120, 150, 180, 240, 300, 360, 420, 480, 540, 600, 2400])
    .groupby(["uniqueCodeL", "time_binned"])
    .agg({"uniqueCodeR": len})
    .reset_index()
    .query("~uniqueCodeR.isnull()")
)

## Output data

In [0]:
lsoa_gp_car_stats_output = (
    lsoa_gp_car_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "gp_name",
        "parent_organisation_code",
        "lat_lon_location",
    ]]
)
lsoa_gp_car_stats_output.head(1)

In [0]:
lsoa_gp_car_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_gp_car_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_gp_car_stats_ICB_output.head(1)

In [0]:
lsoa_gp_car_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_gp_car_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_gp_car_stats_PCON_output.head(1)

In [0]:
lsoa_gp_foot_stats_output = (
    lsoa_gp_foot_stats
    .merge(
        lsoa_to_icb_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )
    .rename(columns={"LSOA21NM_x": "LSOA21NM"})
    .merge(
        lsoa_to_constituency_df,
        how="left",
        left_on="lsoa_id",
        right_on="LSOA21CD",
    )[[
        "lsoa_id",
        "LSOA21NM",
        "Urban_rural_flag",
        "population",
        "lat_lon_lsoa",
        "SICBL23CD",
        "SICBL23CDH",
        "SICBL23NM",
        "ICB23CD",
        "ICB23CDH",
        "ICB23NM",
        "LAD23CD",
        "LAD23NM",
        "PCON24CD",
        "PCON24NM",
        "min_time",
        "distance",
        "number_of_locations_within_5_minutes",
        "number_of_locations_within_10_minutes",
        "number_of_locations_within_15_minutes",
        "number_of_locations_within_20_minutes",
        "number_of_locations_within_30_minutes",
        "min_time_location_id",
        "gp_name",
        "parent_organisation_code",
        "lat_lon_location",
    ]]
)
lsoa_gp_foot_stats_output.head(1)

In [0]:
lsoa_gp_foot_stats_ICB_output = (
    lsoa_to_icb_df[["ICB23CD", "ICB23NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_gp_foot_stats_output,
            "ICB23CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="ICB23CD",
    )
)
lsoa_gp_foot_stats_ICB_output.head(1)

In [0]:
lsoa_gp_foot_stats_PCON_output = (
    lsoa_to_constituency_df[["PCON24CD", "PCON24NM"]]
    .drop_duplicates()
    .merge(
        aggregate_stats_by_geography(
            lsoa_gp_foot_stats_output,
            "PCON24CD",
            [
                "min_time",
                # "distance",
                "number_of_locations_within_5_minutes",
                "number_of_locations_within_10_minutes",
                "number_of_locations_within_15_minutes",
                "number_of_locations_within_20_minutes",
                "number_of_locations_within_30_minutes",
            ],
        ),
        how="left",
        on="PCON24CD",
    )
)
lsoa_gp_foot_stats_PCON_output.head(1)

In [0]:
write_dataset(
    lsoa_gp_car
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_GPs_car_with_ICB_PCON.parquet"
)

In [0]:
write_dataset(
    lsoa_gp_foot
    .merge(
        lsoa_to_icb_df[["LSOA21CD", "ICB23CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD")
    .merge(
        lsoa_to_constituency_df[["LSOA21CD", "PCON24CD"]],
        how="left",
        left_on="uniqueCodeL",
        right_on="LSOA21CD"
    )
    .drop(columns="LSOA21CD"),
    "../../output/full_output_LSOAs_to_GPs_foot_with_ICB_PCON.parquet"
)

In [0]:
write_dataset(
    lsoa_gp_car_stats_output,
    "../../output/summary_data_LSOAs_to_GPs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_gp_foot_stats_output,
    "../../output/summary_data_LSOAs_to_GPs_foot.xlsx"
)

In [0]:
write_dataset(
    lsoa_gp_car_stats_ICB_output,
    "../../output/summary_data_ICBs_to_GPs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_gp_foot_stats_ICB_output,
    "../../output/summary_data_ICBs_to_GPs_foot.xlsx"
)

In [0]:
write_dataset(
    lsoa_gp_car_stats_PCON_output,
    "../../output/summary_data_PCONs_to_GPs_car.xlsx"
)

In [0]:
write_dataset(
    lsoa_gp_car_stats_PCON_output,
    "../../output/summary_data_PCONs_to_GPs_foot.xlsx"
)